# POLITSKY24 Dataset Accessing and Testing

In [2]:
import os
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
#load_dotenv()

# Change path on .env file on your local machine
#data_path = Path(os.getenv("C:/Users/SSHS/Downloads/test"))

In [3]:
# Access the parquet files in the data path

#parquet_files = list(data_path.glob("*.parquet"))
#parquet_files

In [4]:
import polars as pl

# Load each dataset manually into individual variables
user_network_df = pl.read_parquet("user_network_dataset.parquet")
user_post_history_df = pl.read_parquet("user_post_history_dataset.parquet")
user_post_list_df = pl.read_parquet("user_post_list_for_stance_detection.parquet")

# Stance label datasets
llm_full_stance_df = pl.read_parquet("llm_annotated_full_user_stance_dataset.parquet")
llm_val_stance_df = pl.read_parquet("llm_annotated_validation_user_stance_dataset.parquet")
human_val_stance_df = pl.read_parquet("human_annotated_validation_user_stance_dataset.parquet")

# Stance relevancy datasets (Note: surround filename strings in quotes)
human_val_relevancy_post_df = pl.read_parquet(
    "human_annotated_validation_stance_relevancy_dataset (post-target entity pairs).parquet"
)
human_val_relevancy_query_df = pl.read_parquet(
    "human_annotated_validation_stance_relevancy_dataset (query-post stance relevancy pairs).parquet"
)

In [5]:
user_network_df.schema

Schema([('SourceUserId', Int64),
        ('TargetUserId', Int64),
        ('InteractionType', String),
        ('Count', Int64)])

In [6]:
user_network_df

SourceUserId,TargetUserId,InteractionType,Count
i64,i64,str,i64
9,9,"""Like""",1
9,9,"""Quote""",42
9,9,"""Repost""",4
9,10,"""Quote""",2
9,2549,"""Quote""",1
…,…,…,…
401391,398769,"""Like""",2
401391,398831,"""Quote""",1
401391,401391,"""Like""",3


In [7]:
# Check number of rows in user_network_df

user_network_df.shape[0]

1676895

In [8]:
user_post_history_df.schema

Schema([('PostId', Int64),
        ('UserId', Int64),
        ('PostTime', String),
        ('Content', String),
        ('Hashtags', List(String)),
        ('Languages', List(String)),
        ('Mentions', List(Int64)),
        ('IsRepost', Boolean),
        ('SourceUserId', Int64),
        ('IsQuote', Boolean),
        ('QuoteId', Int64),
        ('QuoteUserId', Int64),
        ('QuoteContent', String),
        ('IsReply', Boolean),
        ('ParentId', Int64),
        ('ParentUserId', Int64)])

In [9]:
user_post_history_df.select(pl.all().n_unique()).to_dicts()[0]

{'PostId': 14830231,
 'UserId': 8561,
 'PostTime': 18230517,
 'Content': 13138778,
 'Hashtags': 285770,
 'Languages': 1193,
 'Mentions': 88178,
 'IsRepost': 2,
 'SourceUserId': 217858,
 'IsQuote': 2,
 'QuoteId': 975337,
 'QuoteUserId': 105488,
 'QuoteContent': 910916,
 'IsReply': 2,
 'ParentId': 5472655,
 'ParentUserId': 291653}

In [10]:
# Exact duplicate rows (list columns may need to be dropped/converted first)
user_post_history_df.is_duplicated().sum()

# Rows sharing a PostId
dupes = user_post_history_df.filter(pl.col("PostId").is_duplicated())
dupes.sort("PostId").head(20)

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
18,16171,"""2023-12-03T01:11:35.122Z""","""The end of Moorevember coincid…",[],"[""en""]",[],true,7,false,-1,-1,"""""",false,-1,-1
18,71390,"""2023-12-04T22:26:06.295Z""","""The end of Moorevember coincid…",[],"[""en""]",[],true,7,false,-1,-1,"""""",false,-1,-1
47,9,"""2023-06-22T00:48:37.165Z""","""I used a Chrome/Edge extension…",[],[],[],false,9,false,-1,-1,"""""",false,15691770,343645
47,285903,"""2023-06-22T00:49:34.781Z""","""I used a Chrome/Edge extension…",[],[],[],true,9,false,-1,-1,"""""",false,15691770,343645
279,9,"""2023-07-06T17:04:05.906Z""","""Should be required that PD off…",[],"[""en""]",[],false,9,false,-1,-1,"""""",false,4244240,90714
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
999,141415,"""2023-10-03T01:52:13.292Z""","""Let's talk about how domestic …",[],"[""en""]",[],true,9,false,-1,-1,"""""",false,-1,-1
1000,90924,"""2023-10-03T01:47:26.282Z""","""Going along with Sheryl's post…",[],"[""en""]",[],true,9,false,-1,-1,"""""",true,999,9
1000,9,"""2023-10-03T00:58:13.600Z""","""Going along with Sheryl's post…",[],"[""en""]",[],false,9,false,-1,-1,"""""",true,999,9


In [11]:
user_post_history_df["IsReply"].value_counts()

IsReply,count
bool,u32
false,11509179
true,6907608


In [12]:
# Non-replies that have content in ParentUserId
falseReplies = user_post_history_df.filter(
    (pl.col("IsReply") == "false") & (pl.col("ParentUserId") != -1)
    )
falseReplies.head()

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
10727556,235242,"""2024-11-19T23:11:37.530Z""","""Because a LOT of these people …",[],"[""en""]",[],false,235242,false,-1,-1,"""""",false,12143282,268404
7583443,163361,"""2024-11-19T23:10:37.854Z""","""I'm trying to get used to it""",[],"[""en""]",[],false,163361,false,-1,-1,"""""",false,8475193,183152
8501377,183843,"""2024-11-19T23:08:13.424Z""","""What do you expect from a lose…",[],"[""en""]",[],false,183843,false,-1,-1,"""""",false,3826921,81362
10102388,221773,"""2024-11-19T23:07:03.508Z""","""ty comrade ❤️ op is on some ve…",[],"[""en""]",[],false,221773,false,-1,-1,"""""",false,13234926,290109
8060256,171717,"""2024-11-19T23:05:44.594Z""","""""",[],"[""en""]",[],false,171717,false,-1,-1,"""""",false,17883018,395662


In [13]:
# Non-reposts all have SourceUserIds equal to UserId (original author)
falseRepost = user_post_history_df.filter(
    (pl.col("IsRepost") == "false") & (pl.col("SourceUserId") != -1)
    )
falseRepost.head()

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
7840946,167406,"""2024-11-19T23:12:12.092Z""","""it’s a tragedy of the commons.…",[],"[""en""]",[],false,167406,false,-1,-1,"""""",true,15792425,345622
17079336,377198,"""2024-11-19T23:12:11Z""","""Notícia da @anonymous ""Xi Jin…",[],[],[],false,377198,false,-1,-1,"""""",false,-1,-1
9179962,196837,"""2024-11-19T23:12:11.298Z""","""Have you run it by the Godless…",[],"[""en""]",[],false,196837,false,-1,-1,"""""",true,15458353,336741
2108430,45089,"""2024-11-19T23:12:11.084Z""","""This! This is the energy we ne…",[],"[""en""]",[],false,45089,true,2109954,45144,"""WATCH: “You’re a baby shit.” …",false,-1,-1
4231604,90172,"""2024-11-19T23:12:10Z""","""WATCH: Trending Stories: Gas P…",[],[],[],false,90172,false,-1,-1,"""""",false,-1,-1


In [14]:
falseQuotes = user_post_history_df.filter(
    (pl.col("IsQuote") == "false") & (pl.col("QuoteUserId") != -1)
    )
falseQuotes.head()

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64


In [15]:
user_post_history_df.shape

(18416787, 16)

In [16]:
user_post_list_df.schema

Schema([('UserId', Int64), ('PostIdList', List(Int64))])

In [17]:
llm_full_stance_df.schema

Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [18]:
# Get number of unique values in the stance labbel column
llm_full_stance_df.select(pl.col("StanceLabel").n_unique()).to_dicts

<bound method DataFrame.to_dicts of shape: (1, 1)
┌─────────────┐
│ StanceLabel │
│ ---         │
│ u32         │
╞═════════════╡
│ 3           │
└─────────────┘>

In [19]:
# Get the count of each stance label using the value_counts() method
stance_counts = llm_full_stance_df["StanceLabel"].value_counts()
stance_counts

StanceLabel,count
str,u32
"""Favor""",3071
"""Against""",7832
"""Neither""",5141


In [20]:
llm_full_stance_df.head()

UserId,TargetEntity,Context,LLMAnswerContent,LLMAnswerUsage,SourcePosts,Spans,Reason,StanceLabel,ConfidenceLevel
i64,str,list[str],str,str,list[str],list[str],str,str,f64
9,"""Harris""","[""Assuming Harris is the pick. Biden might have endorsed her but I sure as shit hope there's been arrangements to make her the nominee with no fuss."", ""I mean so do I, there's several people I'd want over Harris. That's...not the same as refusing to vote for or support Harris in THIS election. Someone criticizing Harris for specific positions does not automatically equal ""will not vote for her"""", … ""Jill Stein is trash and cannot be trusted. https://anonymous""]","""{ ""source_tweets"": [ …","""{'input_tokens': 888, 'output_…","[""Assuming Harris is the pick. Biden might have endorsed her but I sure as shit hope there's been arrangements to make her the nominee with no fuss."", ""I mean so do I, there's several people I'd want over Harris. That's...not the same as refusing to vote for or support Harris in THIS election."", … ""Cannot express how badly this whole 'Dump Biden' thing could go if it also pulls in Harris""]","[""hope there's been arrangements to make her the nominee with no fuss"", ""not the same as refusing to vote for or support Harris in THIS election"", … ""Cannot express how badly this whole 'Dump Biden' thing could go if it also pulls in Harris""]","""The user expresses a preferenc…","""Favor""",0.85
9,"""Trump""","[""Gonna fucking Battle Royale us into a Trump presidency"", ""I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement."", … ""Hell I'm still seeing BRUCE SPRINGSTEEN takes that are ""too bad he went woke and liberal!""""]","""{ ""source_tweets"": [ …","""{'input_tokens': 810, 'output_…","[""Gonna fucking Battle Royale us into a Trump presidency"", ""I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement."", … ""Any undecided voter who's paid even the slightest attention is a Trump voter trying to justify their vote.""]","[""Gonna fucking Battle Royale us into a Trump presidency"", ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"", … ""Any undecided voter who's paid even the slightest attention is a Trump voter trying to justify their vote""]","""The tweets express a critical …","""Against""",0.95
10,"""Harris""","[""Kamala/Harris"", ""Kamala Harris is very good at this."", … ""Republicans attacks against Kamala thus far: - She's hot - She enjoys laughing - She's quirky""]","""{ ""source_tweets"": [ …","""{'input_tokens': 803, 'output_…","[""Kamala Harris is very good at this."", ""I think Kamala is going to win."", … ""They need to run against Elon Musk, Peter Thiel, David Sacks and the rest of these white nationalist billionaires who oppose unions, democracy, and anything which gives working people a say.""]","[""Kamala Harris is very good at this."", ""I think Kamala is going to win."", … ""run against Elon Musk, Peter Thiel, David Sacks and the rest of these white nationalist billionaires""]","""The tweets express strong supp…","""Favor""",0.95
10,"""Trump""","[""Not to get too political on here, but I really hope Donald Trump does not become president again."", ""Nothing but respect for my president 🥥🌴🥥🌴"", … ""AP still has Trump up""]","""{ ""source_tweets"": [""Not t…","""{'input_tokens': 808, 'output_…","[""Not to get too political on here, but I really hope Donald Trump does not become president again."", ""Not to get too political on here, but I really hope that Donald Trump does not win the presidential election."", ""If Donald Trump wins the presidential election after going full-fascist, I think that will be quite bad

In [21]:
llm_val_stance_df.schema


Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [22]:
human_val_stance_df.schema

Schema([('UserId', Int64), ('Trump', String), ('Harris', String)])

In [23]:
human_val_relevancy_post_df.schema

Schema([('PostId', Int64), ('Trump', Int64), ('Harris', Int64)])

In [24]:
human_val_relevancy_query_df.schema

Schema([('Query', String),
        ('PostId', Int64),
        ('StanceRelevanceLabel', String)])

## Graph Creation Testing

The code under this section of the notebook is untested and unverified. Double check before being tested.

Simple idea of how we create the data needed for the GNN: User Network Graph creation (likes and reposts) --> Connect this to the LLM full stance using the user id. <br>
    - Verify if they have the same id types (int64) and if they can be connect to each other

In [25]:
import polars as pl

core = user_post_history_df["UserId"].unique()

reposts = (
    user_post_history_df
    .filter(pl.col("IsRepost"))
    .filter(pl.col("SourceUserId").is_in(core)) # keep core users only
    .filter(pl.col("UserId") != pl.col("SourceUserId"))  # drop self-loops
    .group_by("UserId", "SourceUserId")
    .agg(pl.len().alias("weight"))
    .rename({"UserId": "src", "SourceUserId": "dst"})
)

C:\Users\SSHS\AppData\Local\Temp\ipykernel_13340\2874847685.py:8: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("SourceUserId").is_in(core)) # keep core users only


In [26]:
reposts

src,dst,weight
i64,i64,u32
393527,31639,1
364615,13215,69
131119,195196,2
28009,7920,1
364780,336013,1
…,…,…
117720,148578,1
367452,107328,3
219554,239259,1


In [27]:
net = pl.read_parquet("user_network_dataset.parquet")  # use the actual filename/format from Zenodo
likes = (
    net.filter(pl.col("InteractionType") == "Like")  # adjust column names to match the file
    .select(src="SourceUserId", dst="TargetUserId", weight="Count")
)

In [28]:
likes

src,dst,weight
i64,i64,i64
9,9,1
9,13873,1
9,14511,1
9,62997,1
9,92773,3
…,…,…
401391,385828,4
401391,392266,1
401391,394655,1


Also do the same for quotes to test all possible interactions (also replies but would have to manually derive the connections in `user_post_history`)

In [29]:
quotes = (
    net.filter(pl.col("InteractionType") == "Quote")
    .select(src="SourceUserId", dst="TargetUserId", weight="Count")
)

In [30]:
quotes

src,dst,weight
i64,i64,i64
9,9,42
9,10,2
9,2549,1
9,3125,1
9,4109,1
…,…,…
401391,384107,3
401391,384773,1
401391,398501,2


In [37]:
replies = (
    user_post_history_df
    .filter(pl.col("IsReply"))
    .filter(pl.col("ParentUserId").is_in(core))
    .filter(pl.col("UserId") != pl.col("ParentUserId"))
    .group_by("UserId", "ParentUserId")
    .agg(pl.len().alias("weight"))
    .rename({"UserId": "src", "ParentUserId": "dst"})
)

C:\Users\SSHS\AppData\Local\Temp\ipykernel_13340\411258742.py:4: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("ParentUserId").is_in(core))


In [31]:
import torch

nodes = core.sort()
idx = {u: i for i, u in enumerate(nodes.to_list())}

def to_edge_index(edges):
    s = [idx[u] for u in edges["src"]]
    d = [idx[u] for u in edges["dst"]]
    return torch.tensor([s, d]), torch.tensor(edges["weight"].to_list(), dtype=torch.float)

edge_index, edge_weight = to_edge_index(likes)  # or reposts

#### Build graph using NetworkX

In [32]:
# NOTE: For now, the graph construction is separate for each interaction type
# We can convert this to a single multi-relational graph later on if needed

In [33]:
import networkx as nx

def build_graph(df: pl.DataFrame) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_weighted_edges_from(
        df.select("src", "dst", "weight").iter_rows(),
        weight="weight"
    )
    return G

In [62]:
reposts_graph = build_graph(reposts)
likes_graph = build_graph(likes)
quotes_graph = build_graph(quotes)
replies_graph = build_graph(replies)

**Attach stance labels**

In [40]:
llm = pl.read_parquet("llm_annotated_full_user_stance_dataset.parquet")

In [41]:
print(llm["TargetEntity"].value_counts())
print(llm["StanceLabel"].value_counts())
print(llm["ConfidenceLevel"].value_counts())

shape: (2, 2)
┌──────────────┬───────┐
│ TargetEntity ┆ count │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ Trump        ┆ 8022  │
│ Harris       ┆ 8022  │
└──────────────┴───────┘
shape: (3, 2)
┌─────────────┬───────┐
│ StanceLabel ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ Neither     ┆ 5141  │
│ Against     ┆ 7832  │
│ Favor       ┆ 3071  │
└─────────────┴───────┘
shape: (13, 2)
┌─────────────────┬───────┐
│ ConfidenceLevel ┆ count │
│ ---             ┆ ---   │
│ f64             ┆ u32   │
╞═════════════════╪═══════╡
│ null            ┆ 4961  │
│ 0.65            ┆ 2     │
│ 0.7             ┆ 300   │
│ 0.75            ┆ 25    │
│ 0.99            ┆ 49    │
│ …               ┆ …     │
│ 0.6             ┆ 61    │
│ 0.9             ┆ 2003  │
│ 1.0             ┆ 1     │
│ 0.98            ┆ 323   │
│ 0.95            ┆ 7274  │
└─────────────────┴───────┘


In [48]:
llm_stances = (
    llm.select("UserId", "TargetEntity", "StanceLabel")
    .pivot(
        on="TargetEntity",
        index="UserId",
        values=["StanceLabel"],
        aggregate_function="first"
    )
)
llm_stances.head()

UserId,Harris,Trump
i64,str,str
9,"""Favor""","""Against"""
10,"""Favor""","""Against"""
49,"""Against""","""Against"""
105,"""Neither""","""Against"""
144,"""Against""","""Against"""


In [50]:
attrs = {
    r["UserId"]: {"Trump": r["Trump"], "Harris": r["Harris"]}
    for r in llm_stances.iter_rows(named=True)
}

In [63]:
nx.set_node_attributes(reposts_graph, attrs)
nx.set_node_attributes(likes_graph, attrs)
nx.set_node_attributes(quotes_graph, attrs)
nx.set_node_attributes(replies_graph, attrs)

In [ ]:
# Check if all nodes have user stances
G = replies_graph # check node attributes of graphs here
labeled_trump = sum(1 for n in G if G.nodes[n].get("Trump") is not None)
print(f"{labeled_trump}/{G.number_of_nodes()} nodes have a Trump label")

labeled_harris = sum(1 for n in G if G.nodes[n].get("Harris") is not None)
print(f"{labeled_harris}/{G.number_of_nodes()} nodes have a Harris label")

7768/8241 nodes have a Trump label
7768/8241 nodes have a Harris label


Some users did not appear in the stance table (important later on in visualization)

In [ ]:
# NOTE: Visualization for stance labels can get messy since user's stance is based on two attributes
# May need to label into just one (pro-Trump, anti-Trump, etc.)

#### Visualization

In [66]:
# Testing only
"""
import matplotlib.pyplot as plt

G = reposts_graph # select graph here
core = nx.k_core(G.to_undirected(), k=3)
pos = nx.spring_layout(core, k=0.15, iterations=50, seed=42)

colors = ["tab:red" if core.nodes[n].get("Trump") == 1 else "tab:blue" if core.nodes[n].get("Harris") == 1 else "lightgray"
          for n in core]
sizes = [20 + 300 * nx.pagerank(core)[n] for n in core]  # compute PageRank once outside the loop in real code

nx.draw_networkx_edges(core, pos, alpha=0.05, width=0.3)
nx.draw_networkx_nodes(core, pos, node_color=colors, node_size=sizes, linewidths=0)
plt.axis("off"); plt.show()
"""

'\nimport matplotlib.pyplot as plt\n\nG = reposts_graph # select graph here\ncore = nx.k_core(G.to_undirected(), k=3)\npos = nx.spring_layout(core, k=0.15, iterations=50, seed=42)\n\ncolors = ["tab:red" if core.nodes[n].get("Trump") == 1 else "tab:blue" if core.nodes[n].get("Harris") == 1 else "lightgray"\n          for n in core]\nsizes = [20 + 300 * nx.pagerank(core)[n] for n in core]  # compute PageRank once outside the loop in real code\n\nnx.draw_networkx_edges(core, pos, alpha=0.05, width=0.3)\nnx.draw_networkx_nodes(core, pos, node_color=colors, node_size=sizes, linewidths=0)\nplt.axis("off"); plt.show()\n'

NetworkX visualization library can get messy with a large amount of nodes and edges, best use the following recommended libraries, or others not mentioned here:
- gephi (export with `nx.write_gexf(graph, "graph.gexf)`)
- ForceAtlas2 (separate clusters better)
- cosmograph (GPU-accelerated interactable view)